# Austin Metro Economic Productivity Map — Value per Acre

Urban3 / Strong-Towns style **value-per-acre** map: every parcel extruded by its economic value
per acre, so dense central land towers over big-box / suburban tracts.

**Metrics (per parcel):**
- `value_per_acre     = market_value / land_acres`  — what the land is worth
- `value_per_acre_adj = value_per_acre / pvs_ratio` — sales-ratio normalized for cross-county comparability
- `tax_per_acre       = taxable_value * effective_rate / land_acres` — fiscal layer (Travis only)

**Data:** per-county ArcGIS FeatureServers (value+geometry+acreage bundled). Config + fetcher in
`v2_county_sources.py` / `v2_fetch_parcels.py`. Counties: Travis, Williamson, Hays. Verified 2026-06-29.

**Two scopes:** *Slice* (default) = downtown-Austin bbox over Travis (fast, true-parcel 3D hero);
*Metro* = all three counties from cached parquet, rendered as H3 hexes. Set `LOAD_METRO_CACHE = True`.

**Color:** quantile-binned **turbo** palette (blue=low → green → yellow → red=high) so the full
range is used and differences read clearly; light labeled basemap + bold major-highway overlay
for geographic orientation.

## Setup

In [1]:
import sys, math
from pathlib import Path
import json, urllib.request, urllib.parse

import geopandas as gpd
import pandas as pd
import numpy as np
import pydeck as pdk
import folium
import h3
import matplotlib
import branca.colormap as bcm

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'report_pipeline'))
from v2_county_sources import COUNTY_SOURCES, PVS_RATIOS
from v2_fetch_parcels import fetch_county_gdf

OUT_DIR = REPO_ROOT / 'outputs'; OUT_DIR.mkdir(exist_ok=True)
PROC_DIR = REPO_ROOT / 'processed_data'; PROC_DIR.mkdir(exist_ok=True)
METRO_PARQUET = PROC_DIR / 'parcels_value_per_acre_metro.parquet'
HWY_GEOJSON = PROC_DIR / 'metro_highways.geojson'
print('counties:', list(COUNTY_SOURCES), '| pvs:', PVS_RATIOS)

counties: ['travis', 'williamson', 'hays'] | pvs: {'travis': 1.0, 'williamson': 0.96, 'hays': 0.97}


## Config

In [2]:
# --- scope ---
DOWNTOWN_BBOX = (-97.78, 30.24, -97.71, 30.30)
SCOPE = {'travis': DOWNTOWN_BBOX}
LOAD_METRO_CACHE = False

# --- cleaning ---
MIN_ACRES = 0.01
EFFECTIVE_TAX_RATE = 0.021

# --- rendering ---
MAX_HEIGHT = 2500.0
ELEV_METRIC = 'value_per_acre'   # or 'value_per_acre_adj' / 'tax_per_acre'
SIMPLIFY_M = 2.0
H3_RES = 8
N_BINS = 8                       # quantile color classes
CMAP = 'turbo'                   # matplotlib colormap name (turbo/viridis/Spectral_r/plasma)
BASEMAP = 'CartoDB positron'     # light, labeled -> 'see what is where'

## Major highways (for orientation)
Loaded from cache, or fetched from OpenStreetMap (motorway + trunk) and cached.

In [3]:
def load_highways(bbox=(29.55, -98.30, 30.95, -97.30)):
    if HWY_GEOJSON.exists():
        return gpd.read_file(HWY_GEOJSON)
    S, W, N, E = bbox
    q = f'[out:json][timeout:90];(way["highway"~"^(motorway|trunk)$"]({S},{W},{N},{E}););out geom;'
    for ep in ['https://overpass-api.de/api/interpreter', 'https://overpass.kumi.systems/api/interpreter']:
        try:
            url = ep + '?' + urllib.parse.urlencode({'data': q})
            req = urllib.request.Request(url, headers={'User-Agent': 'fire-incident-analysis/1.0'})
            data = json.load(urllib.request.urlopen(req, timeout=120)); break
        except Exception as e:
            print('overpass fail', ep, e); data = None
    feats = []
    for e in (data or {}).get('elements', []):
        if e.get('geometry'):
            coords = [[p['lon'], p['lat']] for p in e['geometry']]
            if len(coords) >= 2:
                feats.append({'type': 'Feature', 'properties': {'ref': e.get('tags', {}).get('ref', '')},
                              'geometry': {'type': 'LineString', 'coordinates': coords}})
    fc = {'type': 'FeatureCollection', 'features': feats}
    json.dump(fc, open(HWY_GEOJSON, 'w'))
    return gpd.read_file(HWY_GEOJSON)

hwy = load_highways()
hwy['geometry'] = hwy.to_crs(2277).geometry.simplify(150).to_crs(4326)  # lighten for web
print(f'{len(hwy)} highway segments')

3736 highway segments


## Fetch (or load cached metro)

In [4]:
if LOAD_METRO_CACHE:
    print('loading metro parquet:', METRO_PARQUET)
    g = gpd.read_parquet(METRO_PARQUET); PRECOMPUTED = True
else:
    frames = []
    for county, bbox in SCOPE.items():
        print(f'fetching {county}' + (f' bbox {bbox}' if bbox else ' (full county)') + ' ...')
        gc = fetch_county_gdf(county, bbox=bbox, only_valued=True, with_geometry=True)
        print(f'  {len(gc):,} parcels'); frames.append(gc)
    g = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs='EPSG:4326'); PRECOMPUTED = False
print(f'{len(g):,} parcels')

fetching travis bbox (-97.78, 30.24, -97.71, 30.3) ...


  21,092 parcels
21,092 parcels


## Clean + compute metrics
**Acreage:** suburban CADs leave `land_acres` null for ~60% of platted lots, so we fall back to the
parcel's **geometry footprint** (area in EPSG:2277) wherever the CAD attribute is null/<=0 — recovering
the suburbs and unifying the counties. (Skipped when loading the cached parquet, already clean.)

In [5]:
if not PRECOMPUTED:
    g = g[g.geometry.notna()].copy()
    g['market_value'] = pd.to_numeric(g['market_value'], errors='coerce')
    attr = pd.to_numeric(g['land_acres'], errors='coerce')
    geom_ac = (g.to_crs(2277).geometry.area / 43560.0).values
    g['acre_source'] = np.where(attr > 0, 'attr', 'geom')
    g['land_acres'] = attr.where(attr > 0, geom_ac)
    before = len(g)
    g = g[(g['land_acres'] >= MIN_ACRES) & (g['market_value'] > 0)]
    print(f'dropped {before - len(g):,} below {MIN_ACRES} ac / zero value; '
          f"acre source: {g['acre_source'].value_counts().to_dict()}")
    g['value_per_acre'] = g['market_value'] / g['land_acres']
    g['pvs_ratio'] = g['county'].map(PVS_RATIOS)
    g['value_per_acre_adj'] = g['value_per_acre'] / g['pvs_ratio']
    g['taxable_value'] = pd.to_numeric(g.get('taxable_value'), errors='coerce')
    g['tax_per_acre'] = g['taxable_value'] * EFFECTIVE_TAX_RATE / g['land_acres']
    g = g[g['value_per_acre'].between(1, 1e12)]
print(f'{len(g):,} parcels after cleaning')
g[['value_per_acre', 'value_per_acre_adj', 'tax_per_acre']].describe()

dropped 9 below 0.01 ac / zero value; acre source: {'attr': 21083}
21,083 parcels after cleaning


,value_per_acre,value_per_acre_adj,tax_per_acre
count,2.108300e+04,2.108300e+04,2.108200e+04
mean,7.644454e+06,7.644454e+06,1.605347e+05
std,1.428617e+07,1.428617e+07,3.000166e+05
min,1.296092e+01,1.296092e+01,2.721794e-01
25%,3.876747e+06,3.876747e+06,8.141116e+04
50%,5.439956e+06,5.439956e+06,1.142369e+05
75%,8.065518e+06,8.065518e+06,1.693817e+05
max,4.836484e+08,4.836484e+08,1.015662e+07


## Validation — the Urban3 sanity story
Top by value/acre = dense central commercial/residential; bottom = large low-value tracts & fringe land.

In [6]:
cols = [c for c in ['county','parcel_id','land_use','market_value','land_acres','value_per_acre'] if c in g.columns]
print('TOP 10 by value/acre'); display(g.sort_values('value_per_acre', ascending=False)[cols].head(10))
print('BOTTOM 10 by value/acre'); display(g.sort_values('value_per_acre')[cols].head(10))

TOP 10 by value/acre


,county,parcel_id,land_use,market_value,land_acres,value_per_acre
4970,travis,0206011606,OFF HI-RISE >= 6,196119412,0.4055,4.836484e+08
19202,travis,0206011205,LUXURY HI-RISE APTS 100+,192780000,0.4055,4.754131e+08
5009,travis,0205021001,HIRISE CONDO/APT,188356279,0.4055,4.645038e+08
20556,travis,0203031001,LUXURY HI-RISE APTS 100+,233950000,0.5405,4.328400e+08
20013,travis,0206011901,OFF HI-RISE >= 6,387556329,0.9377,4.133052e+08
20249,travis,0206030709,HOTEL-FULL SERVC,98000000,0.2535,3.865878e+08
14965,travis,0206030816,HOTEL-FULL SERVC,58000000,0.1593,3.640929e+08
3277,travis,0214011303,APARTMENT 100+,143490000,0.4014,3.574738e+08
1512,travis,0210021714,OFF HI-RISE >= 6,147991225,0.4406,3.358857e+08
12150,travis,0105001001,OFF HI-RISE >= 6,275633332,0.8230,3.349129e+08


BOTTOM 10 by value/acre


,county,parcel_id,land_use,market_value,land_acres,value_per_acre
17112,travis,0104090221,NaN,20,1.5431,12.960923
18099,travis,0217130103,NaN,20000,30.1221,663.964332
18103,travis,0219120242,NaN,10000,13.1408,760.988676
14167,travis,0215080169,NaN,210,0.0483,4347.826087
10539,travis,NaN,RETAIL STORE,750000,163.6395,4583.245488
20589,travis,0105000103,NaN,750000,163.6395,4583.245488
16410,travis,0214000102,NaN,1500,0.3000,5000.000000
20971,travis,0405061601,NaN,26250,3.9209,6694.891479
10934,travis,0404070326,NaN,5478,0.6130,8936.378467
16176,travis,0404070615,Detail Only,7699,0.7708,9988.323819


## Color: quantile-binned turbo
Equal numbers of hexes/parcels per color class, so the palette spans the data and small differences
stay visible. `step` doubles as the map legend; `rgb_of` feeds pydeck's RGB fill.

In [7]:
def make_step(series, n=N_BINS, cmap_name=CMAP, caption='value per acre ($)'):
    vals = series[series > 0]
    edges = list(np.unique(np.quantile(vals, np.linspace(0, 1, n + 1))))
    cols = [matplotlib.colors.to_hex(matplotlib.colormaps[cmap_name](x))
            for x in np.linspace(0.04, 0.96, len(edges) - 1)]
    return bcm.StepColormap(cols, index=edges, vmin=edges[0], vmax=edges[-1], caption=caption)

def rgb_of(step, v):
    h = step(v).lstrip('#')
    return [int(h[i:i+2], 16) for i in (0, 2, 4)]

## 3D true-parcel scene (slice / bounded scope)
pydeck `PolygonLayer`, extruded by `ELEV_METRIC`, colored by quantile turbo. True parcels at full
metro scale break the browser, so this auto-skips above 200k parcels (use the H3 metro scene instead).

In [8]:
POLY_LIMIT = 200_000
if len(g) > POLY_LIMIT:
    print(f'{len(g):,} parcels > {POLY_LIMIT:,}: skipping true-parcel 3D (use H3 metro scene).')
    out_html_3d = None
else:
    gg = g.copy()
    if SIMPLIFY_M > 0:
        gg['geometry'] = gg.to_crs(2277).geometry.simplify(SIMPLIFY_M).to_crs(4326)
    step3 = make_step(gg[ELEV_METRIC])
    p_hi = gg[ELEV_METRIC].quantile(0.99); elev_scale = MAX_HEIGHT / p_hi
    def rings(geom):
        if geom.geom_type == 'Polygon': return [list(geom.exterior.coords)]
        if geom.geom_type == 'MultiPolygon': return [list(p.exterior.coords) for p in geom.geoms]
        return []
    poly_data = []
    for _, r in gg.iterrows():
        v = r[ELEV_METRIC]; color = rgb_of(step3, v); v_clip = min(v, p_hi)
        for ring in rings(r.geometry):
            poly_data.append({'polygon': [[x, y] for x, y in ring], 'elevation': v_clip * elev_scale,
                'color': color, 'vpa': f"${r['value_per_acre']:,.0f}/acre",
                'mkt': f"${r['market_value']:,.0f}", 'acres': f"{r['land_acres']:.3f}"})
    hwy_paths = [{'path': list(geom.coords)} for geom in hwy.geometry if geom.geom_type == 'LineString']
    cx, cy = float(gg.geometry.union_all().centroid.x), float(gg.geometry.union_all().centroid.y)
    layers = [pdk.Layer('PolygonLayer', poly_data, get_polygon='polygon', get_elevation='elevation',
        get_fill_color='color', extruded=True, pickable=True, elevation_scale=1),
        pdk.Layer('PathLayer', hwy_paths, get_path='path', get_color=[20, 20, 20],
        width_min_pixels=2, get_width=3)]
    deck = pdk.Deck(layers=layers,
        initial_view_state=pdk.ViewState(latitude=cy, longitude=cx, zoom=13, pitch=55, bearing=20),
        map_style='road', tooltip={'text': '{vpa}\nMarket: {mkt}\nAcres: {acres}'})
    out_html_3d = OUT_DIR / 'map_value_per_acre_3d.html'
    deck.to_html(str(out_html_3d), notebook_display=False)
    print(f'{len(poly_data):,} rings -> {out_html_3d}')

21,223 rings -> /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_3d.html


## H3 hex aggregation (metro-scale)
Bin parcels to H3 cells by centroid; per hex `value_per_acre = sum(market_value) / sum(land_acres)`.

In [9]:
cent = g.to_crs(2277).geometry.centroid.to_crs(4326)
hexes = [h3.latlng_to_cell(y, x, H3_RES) for x, y in zip(cent.x.values, cent.y.values)]
agg = pd.DataFrame({'hex': hexes, 'market_value': g['market_value'].values, 'land_acres': g['land_acres'].values})
hx = agg.groupby('hex').agg(market_value=('market_value','sum'), land_acres=('land_acres','sum'),
                            n=('hex','size')).reset_index()
hx['value_per_acre'] = hx['market_value'] / hx['land_acres']
print(f'{len(hx):,} H3 cells (res {H3_RES}) covering {len(g):,} parcels')
hx['value_per_acre'].describe()

75 H3 cells (res 8) covering 21,083 parcels


count    7.500000e+01
mean     7.664406e+06
std      8.091287e+06
min      4.583245e+03
25%      4.075881e+06
50%      5.374669e+06
75%      7.090706e+06
max      4.303018e+07
Name: value_per_acre, dtype: float64

### 2D shareable map (folium choropleth + highways + legend)

In [10]:
step = make_step(hx['value_per_acre'])
clat = float(cent.y.mean()); clon = float(cent.x.mean())
m = folium.Map(location=[clat, clon], zoom_start=11, tiles=BASEMAP)
# hex choropleth
for _, r in hx.iterrows():
    boundary = h3.cell_to_boundary(r['hex'])
    folium.Polygon([[lat, lng] for lat, lng in boundary], color='#666', weight=0.3,
        fill=True, fill_color=step(r['value_per_acre']), fill_opacity=0.7,
        tooltip=f"${r['value_per_acre']:,.0f}/acre · {int(r['n'])} parcels").add_to(m)
# major highways: white casing + dark core so they read over the choropleth
hwy_gj = hwy.to_json()
folium.GeoJson(hwy_gj, style_function=lambda f: {'color': '#ffffff', 'weight': 4.5, 'opacity': 0.9}).add_to(m)
folium.GeoJson(hwy_gj, style_function=lambda f: {'color': '#1a1a1a', 'weight': 1.8, 'opacity': 0.95}).add_to(m)
step.add_to(m)  # legend
out_html_2d = OUT_DIR / 'map_value_per_acre_metro.html'
m.save(str(out_html_2d)); print('wrote', out_html_2d)

wrote /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_metro.html


### 3D extruded hexes (metro hero scene)

In [11]:
p_hi_h = hx['value_per_acre'].quantile(0.99); elev_scale_h = MAX_HEIGHT / p_hi_h
hx3 = hx.copy()
hx3['elevation'] = hx3['value_per_acre'].clip(upper=p_hi_h) * elev_scale_h
hx3['color'] = hx3['value_per_acre'].apply(lambda v: rgb_of(step, v))
hx3['vpa'] = hx3['value_per_acre'].apply(lambda v: f'${v:,.0f}/acre')
hwy_paths = [{'path': list(geom.coords)} for geom in hwy.geometry if geom.geom_type == 'LineString']
layers = [pdk.Layer('H3HexagonLayer', hx3, get_hexagon='hex', get_elevation='elevation',
        get_fill_color='color', extruded=True, pickable=True, elevation_scale=1, coverage=0.95),
        pdk.Layer('PathLayer', hwy_paths, get_path='path', get_color=[15, 15, 15],
        width_min_pixels=2, get_width=4)]
deck_h = pdk.Deck(layers=layers,
    initial_view_state=pdk.ViewState(latitude=clat, longitude=clon, zoom=10, pitch=50, bearing=15),
    map_style='road', tooltip={'text': '{vpa}'})
out_html_metro_3d = OUT_DIR / 'map_value_per_acre_metro_3d.html'
deck_h.to_html(str(out_html_metro_3d), notebook_display=False); print('wrote', out_html_metro_3d)

wrote /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_metro_3d.html


## Outputs
- `outputs/map_value_per_acre_3d.html` — true-parcel 3D (slice)
- `outputs/map_value_per_acre_metro.html` — 2D H3 choropleth + highways + legend (shareable)
- `outputs/map_value_per_acre_metro_3d.html` — 3D extruded H3 hexes + highways (metro hero)

Full metro: build the parquet, set `LOAD_METRO_CACHE = True`, re-run. Tune `CMAP`/`N_BINS`/`BASEMAP`
for color; switch `ELEV_METRIC='value_per_acre_adj'` for the cross-county-comparable layer.